# Multi-period (s, S) inventory policy via dynamic programming

Course 21AIE321T — Stochastic Decision Making

The shop reviews inventory every period and follows the optimal
`(s, S)` policy: order `S − IP` whenever the inventory position
`IP ≤ s`, otherwise order nothing.

Model: single item, backorders, lead time `L`, costs `K, c, h, b`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from demand import from_negbin, from_empirical, fit_from_data
from dp import solve_dp, base_stock_level
from simulate import simulate, ss_policy, base_stock_policy, summarise
from plot import plot_value_and_policy, plot_simulation

## 1. Demand distribution

In [ ]:
dd = from_negbin(mean=100, cv=0.20)   # or fit_from_data('daily_sales.csv')
print(dd.summary())
print('base-stock reference (K=0): S_bs =', base_stock_level(dd, h=3, b=40))

## 2. Solve the (s, S) DP

In [ ]:
K, c, h, b = 150.0, 50.0, 3.0, 40.0
pol = solve_dp(dd, K, c, h, b, gamma=0.99, verbose=True)
print(f'\n(s, S) = ({pol.s}, {pol.S})')

## 3. Value function and policy

In [ ]:
plot_value_and_policy(pol, pol.s, pol.S)
plt.show()

## 4. Simulate and compare

In [ ]:
rng = np.random.default_rng(42)
common = dict(n_periods=20_000, K=K, c=c, h=h, b=b, lead_time=1,
              rng=rng, init_ip=pol.S)

policies = {
    f'(s,S) DP (s={pol.s}, S={pol.S})': ss_policy(pol.s, pol.S),
    f'base-stock S={base_stock_level(dd, h, b)}': base_stock_policy(base_stock_level(dd, h, b)),
}
import pandas as pd
rows = []
dfs = {}
for name, fn in policies.items():
    df = simulate(fn, dd, **common)
    dfs[name] = df
    rows.append(summarise(df, name))
pd.DataFrame(rows).round(2)

In [ ]:
plot_simulation(dfs[next(iter(dfs))])
plt.show()

## 5. Sensitivity of (s, S) to K

In [ ]:
import warnings
Ks = [10, 50, 100, 200, 400, 800, 1500]
ss = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for K_ in Ks:
        p = solve_dp(dd, K_, c, h, b, gamma=0.99, max_iter=2000)
        ss.append((K_, p.s, p.S))
for K_, s_, S_ in ss:
    print(f'K={K_:>5}  s={s_:>5}  S={S_:>5}  S-s={S_-s_:>5}')

## Notes

- `S` is close to the base-stock level `S_bs`; `S − s` grows like `sqrt(2 K μ / h)`
  as `K` grows.
- The DP is solved under **backorders**; the simulator also uses backorders.
- For lead time `L > 1` the policy's shape is preserved but the reported cost is
  the true simulated cost under that `L`, so the comparison is honest.